# 03 · Your First Forecast

The absolute minimum: load the model, feed one array, get a forecast back.
We use synthetic data so this notebook is fully self-contained.

In [ ]:
import torch
import numpy as np
import timesfm

torch.set_float32_matmul_precision("high")

# Downloads ~800 MB of weights the first time, then caches in ~/.cache/huggingface/
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)
print("Model loaded and compiled.")

## Create a simple synthetic series (trend + seasonality)

In [ ]:
rng = np.random.default_rng(0)
t = np.arange(300)
series = (
    0.05 * t                                 # upward trend
    + 10 * np.sin(2 * np.pi * t / 30)        # monthly-ish seasonality
    + rng.normal(0, 1.5, size=t.size)        # noise
).astype(np.float32)

print("series length:", series.shape)

## Forecast the next 24 steps

In [ ]:
horizon = 24
point_forecast, quantile_forecast = model.forecast(
    horizon=horizon,
    inputs=[series],           # a LIST of 1-D arrays (one per series)
)

print("point_forecast   :", point_forecast.shape)      # (1, 24)
print("quantile_forecast:", quantile_forecast.shape)   # (1, 24, 10)
print("\nnext 5 predicted values:", point_forecast[0, :5].round(2))

## Understanding the output

`model.forecast()` returns **two** arrays:

| Array | Shape | Meaning |
| ----- | ----- | ------- |
| `point_forecast` | `(n_series, horizon)` | the median (0.5 quantile) point forecast |
| `quantile_forecast` | `(n_series, horizon, 10)` | probabilistic bands |

The last axis of `quantile_forecast` has **10 slices**:

| Index | Quantile | Use |
| ----- | -------- | --- |
| `0` | mean | average prediction (NOT q0!) |
| `1` | 0.1 | lower bound of the 80% interval |
| `2` | 0.2 | lower bound of the 60% interval |
| `5` | 0.5 | median (== `point_forecast`) |
| `8` | 0.8 | upper bound of the 60% interval |
| `9` | 0.9 | upper bound of the 80% interval |

> ⚠️ **Common mistake:** index `0` is the **mean**, not the 0th percentile.
> q10 lives at index `1`, q90 at index `9`.

In [ ]:
# Handy constants for indexing the quantile axis
IDX_MEAN = 0
IDX_Q10  = 1   # lower bound, 80% interval
IDX_Q20  = 2   # lower bound, 60% interval
IDX_Q50  = 5   # median
IDX_Q80  = 8   # upper bound, 60% interval
IDX_Q90  = 9   # upper bound, 80% interval

## Plot it

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; remove for interactive notebooks
import matplotlib.pyplot as plt

hist = series[-100:]
x_fc = range(len(hist), len(hist) + horizon)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(len(hist)), hist, label="History", color="tab:blue")
ax.plot(x_fc, point_forecast[0], label="Forecast", color="tab:orange")
ax.fill_between(x_fc,
                quantile_forecast[0, :, IDX_Q10],
                quantile_forecast[0, :, IDX_Q90],
                alpha=0.25, color="tab:orange", label="80% interval")
ax.legend(); ax.set_title("TimesFM — first forecast")
fig.tight_layout()
fig.savefig("first_forecast.png", dpi=120)
print("saved first_forecast.png")

🎉 That's the whole idea. Next: a realistic single series with nicer plots.